In [5]:
import pandas as pd
import time
import mlflow
import joblib
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, accuracy_score, f1_score, confusion_matrix

mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_LogisticRegression")

<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/mlruns/2'), creation_time=1777135335016, experiment_id='2', last_update_time=1777135335016, lifecycle_stage='active', name='Classification_LogisticRegression', tags={}, workspace='default'>

I will remove the variable "diabetes_stage" from features to train the model because it will give away if a pacient has diabetes or not.

In [6]:
df = pd.read_csv("../../data/diabetes_dataset_new_variables.csv")

categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level','employment_status', 'age_groups', 'weight_status', "income_level"]
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

In [7]:
for s_name, scaler in scalers.items():
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

    # ---  BASELINE  ---
    with mlflow.start_run(run_name=f"LogReg_{s_name}_Baseline"):
        start_time = time.time()
        model = LogisticRegression(random_state=42)
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time

        y_pred = model.predict(X_test_scaled)
        
        # Log dos parâmetros padrão para o MLflow
        mlflow.log_params(model.get_params()) # Parametros default do modelo LogisticRegression
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "none")
        
        # Log das métricas
        mlflow.log_metric("recall", recall_score(y_test, y_pred))
        mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
        mlflow.log_metric("f1", f1_score(y_test, y_pred))
        mlflow.log_metric("fit_time", duration)

    # --- GRIDSEARCHCV (Otimizando Solver, C e Penalty) ---
    with mlflow.start_run(run_name=f"LogReg_{s_name}_GridSearch"):
        # Lista de dicionários para evitar combinações impossíveis (ex: lbfgs com l1)
        param_grid = [
            {
                'solver': ['lbfgs'],
                'penalty': ['l2'], 
                'C': [0.1, 1.0, 10.0],
                'max_iter': [1000]
            },
            {
                'solver': ['saga'],
                'penalty': ['l1', 'l2'], 
                'C': [0.1, 1.0, 10.0],
                'max_iter': [2000] 
            }
        ]
        
        grid = GridSearchCV(LogisticRegression(random_state=42), param_grid, cv=5, scoring='recall', n_jobs=-1)
        
        start_time = time.time()
        grid.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        # Log no MLflow
        mlflow.log_params(grid.best_params_)
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "GridSearchCV")
        mlflow.log_metric("recall", recall_score(y_test, grid.best_estimator_.predict(X_test_scaled)))
        mlflow.log_metric("accuracy", accuracy_score(y_test, grid.best_estimator_.predict(X_test_scaled)))
        mlflow.log_metric("f1", f1_score(y_test, grid.best_estimator_.predict(X_test_scaled)))
        mlflow.log_metric("fit_time", duration)

    # --- OPTUNA (Otimizando Solver, C e Penalty) ---
    def objective(trial):
        solver_val = trial.suggest_categorical("solver", ["lbfgs", "saga"])
        
        if solver_val == "lbfgs":
            penalty_val = "l2"
        else:
            penalty_val = trial.suggest_categorical("penalty", ["l1", "l2"])
            
        c_val = trial.suggest_float("C", 1e-3, 100, log=True)
        
        model_opt = LogisticRegression(
            solver=solver_val, 
            penalty=penalty_val, 
            C=c_val, 
            max_iter=2000, 
            random_state=42
        )
        
        model_opt.fit(X_train_scaled, y_train)
        return recall_score(y_test, model_opt.predict(X_test_scaled))
    
    with mlflow.start_run(run_name=f"LogReg_{s_name}_Optuna"):
        study = optuna.create_study(direction="maximize")
        start_time = time.time()
        study.optimize(objective, n_trials=15) 
        duration = time.time() - start_time
        
        best_lr = LogisticRegression(**study.best_params, max_iter=2000, random_state=42)
        best_lr.fit(X_train_scaled, y_train)
        
        mlflow.log_params(study.best_params)
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "optuna")
        mlflow.log_metric("recall", study.best_value)
        mlflow.log_metric("fit_time", duration)
        mlflow.log_metric("accuracy", accuracy_score(y_test, best_lr.predict(X_test_scaled)))
        mlflow.log_metric("f1", f1_score(y_test, best_lr.predict(X_test_scaled)))


[I 2026-04-25 18:03:18,210] A new study created in memory with name: no-name-6cbb1025-d126-4e97-ab18-ec09dac8c85f
[I 2026-04-25 18:03:23,762] Trial 0 finished with value: 0.8921412396209008 and parameters: {'solver': 'saga', 'penalty': 'l1', 'C': 5.063163916325861}. Best is trial 0 with value: 0.8921412396209008.
[I 2026-04-25 18:03:24,034] Trial 1 finished with value: 0.8923928541474461 and parameters: {'solver': 'lbfgs', 'C': 0.08933818372689561}. Best is trial 1 with value: 0.8923928541474461.
[I 2026-04-25 18:03:24,748] Trial 2 finished with value: 0.8923928541474461 and parameters: {'solver': 'saga', 'penalty': 'l2', 'C': 0.021062194302088104}. Best is trial 1 with value: 0.8923928541474461.
[I 2026-04-25 18:03:25,632] Trial 3 finished with value: 0.8915541390589616 and parameters: {'solver': 'saga', 'penalty': 'l1', 'C': 0.023307413107679444}. Best is trial 1 with value: 0.8923928541474461.
[I 2026-04-25 18:03:25,857] Trial 4 finished with value: 0.8926444686739915 and parameters

## Runs do Logistic Regression

### 1. LogReg_Standardization_Baseline
**Parâmetros utilizados**
- `scaler`: Standardization
- `optimization`: none
- `solver`: lbfgs
- `penalty`: l2
- `C`: 1.0
- `max_iter`: 100
- `random_state`: 42
- `fit_intercept`: True
- `class_weight`: None
- `dual`: False
- `tol`: 0.0001
- `verbose`: 0
- `warm_start`: False

**Métricas**
- `recall`: 0.8919734966032039
- `accuracy`: 0.8577
- `f1`: 0.8819870625310997
- `fit_time`: 0.3063085079193115 s

### 2. LogReg_Standardization_GridSearch
**Parâmetros utilizados**
- `scaler`: Standardization
- `optimization`: GridSearchCV
- `solver`: saga
- `penalty`: l1
- `C`: 0.1
- `max_iter`: 2000
- `random_state`: 42

**Métricas**
- `recall`: 0.8917218820766586
- `accuracy`: 0.85755
- `f1`: 0.8818479658275619
- `fit_time`: 26.320055723190308 s

### 3. LogReg_Standardization_Optuna
**Parâmetros utilizados**
- `scaler`: Standardization
- `optimization`: optuna
- `solver`: saga
- `penalty`: l2
- `C`: 0.0011678781118622424
- `max_iter`: 2000
- `random_state`: 42

**Métricas**
- `recall`: 0.8941541558332634
- `accuracy`: 0.85625
- `f1`: 0.88118361780386
- `fit_time`: 12.575929641723633 s

### 4. LogReg_Normalization_Baseline
**Parâmetros utilizados**
- `scaler`: Normalization
- `optimization`: none
- `solver`: lbfgs
- `penalty`: l2
- `C`: 1.0
- `max_iter`: 100
- `random_state`: 42
- `fit_intercept`: True
- `class_weight`: None
- `dual`: False
- `tol`: 0.0001
- `verbose`: 0
- `warm_start`: False

**Métricas**
- `recall`: 0.8924767256562945
- `accuracy`: 0.85695
- `f1`: 0.8814977426169076
- `fit_time`: 0.403186559677124 s

### 5. LogReg_Normalization_GridSearch
**Parâmetros utilizados**
- `scaler`: Normalization
- `optimization`: GridSearchCV
- `solver`: lbfgs
- `penalty`: l2
- `C`: 0.1
- `max_iter`: 1000
- `random_state`: 42

**Métricas**
- `recall`: 0.8938186697978696
- `accuracy`: 0.8572
- `f1`: 0.881836988001655
- `fit_time`: 30.38247847557068 s

### 6. LogReg_Normalization_Optuna
**Parâmetros utilizados**
- `scaler`: Normalization
- `optimization`: optuna
- `solver`: lbfgs
- `penalty`: l2
- `C`: 0.0011178096689315107
- `max_iter`: 2000
- `random_state`: 42

**Métricas**
- `recall`: 0.9174704352931309
- `accuracy`: 0.80845
- `f1`: 0.850986036018515
- `fit_time`: 31.633457899093628 s